In [11]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [12]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.split_cp import SplitConformalPredictor

Load data

In [13]:
input_points, output_points = load_data("friedman1")

In [14]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)
(
    proper_train_input_points,
    calib_input_points,
    proper_train_output_points,
    calib_output_points,
) = train_test_split(train_input_points, train_output_points, random_state=0)

Instantiate predictor

In [15]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [16]:
predictor = KernelRegression(
    lam=0.5,
    kernel="laplacian",
    solver="lbfgs", loss_name=loss_name, loss_params=loss_params
)
predictor.fit(proper_train_input_points, proper_train_output_points)

Instantiate region predictor

In [17]:
conformal_predictor = SplitConformalPredictor(predictor, non_conformity_name="absolute")
conformal_predictor.fit(calib_input_points, calib_output_points)
region_predictor = conformal_predictor.predict(test_input_points)

In [18]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [19]:
coverage = np.mean(
    [
        test_output_point in prediction_region
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage)

test coverage:  0.912


In [20]:
prediction_regions

[[array([-1.78381949]),array([1.77003907])],
 [array([-1.78302778]),array([1.77083077])],
 [array([-1.78409423]),array([1.76976432])],
 [array([-1.7836834]),array([1.77017516])],
 [array([-1.78179659]),array([1.77206196])],
 [array([-1.78122855]),array([1.77263])],
 [array([-1.78177779]),array([1.77208077])],
 [array([-1.77927023]),array([1.77458833])],
 [array([-1.78309917]),array([1.77075938])],
 [array([-1.78126861]),array([1.77258994])],
 [array([-1.78175285]),array([1.7721057])],
 [array([-1.78186219]),array([1.77199637])],
 [array([-1.78472706]),array([1.76913149])],
 [array([-1.78161788]),array([1.77224067])],
 [array([-1.78210657]),array([1.77175198])],
 [array([-1.7835712]),array([1.77028736])],
 [array([-1.78251343]),array([1.77134513])],
 [array([-1.78213387]),array([1.77172469])],
 [array([-1.78287935]),array([1.77097921])],
 [array([-1.78268507]),array([1.77117348])],
 [array([-1.78203624]),array([1.77182232])],
 [array([-1.78116087]),array([1.77269768])],
 [array([-1.7864